## Data correction

This notebook helps preparing data obtained from pyridine adsorption IR measurements for further evaluation. The data is extracted from the measurement files, corrected, truncated and saved to new csv files.
The baseline correction is done with the measurement data of the unloaded sample ("_hydrated_"). As only the ring breathing vibrations in the range of 1400 cm$^{-1}$ < $\nu$ < 1600 cm$^{-1}$ are of interest, the data is truncated accordingly.

***

First of all, import all needed packages.

In [ ]:
from pathlib import Path

from modules.data_processer import IRDataHandler
from modules.peak_fitter import PeakFitter, FitConfig


print("All done.")

---
In your current working directory (`cwd`), specify the `folder` containing the data which are to be processed. The list of available files gives an overview over all samples in this specific folder.

In [ ]:
# Setup paths and configuration
folder = "data/PyrIR_OMAS_1-5_T80"
cwd = Path.cwd()
path_to_directory = cwd / folder
print(path_to_directory)
# Initialize data handler
data_handling = IRDataHandler(path_to_directory=path_to_directory, folder = folder, decimal=",")
data_handling.available_files()

---
You can plot all measurement data into one figure for both comparison and publication.
The legend can be extracted from the respective characters in the file name. `val1` gives the first chracter, `val2` the last.

In [ ]:
# Process and plot data
data_handling.get_plot()

If the plots are satisfactory, you can save your corrected data to new csv files with the following command.

In [4]:
data_handling.save_data_to_csv()

In [ ]:
# Save metadata
data_handling.add_sample_metadata(
    sample_mass = 0.0874,
    sample_length = 1.974,
    sample_width=1.061,
    extinction_coefficient_bronsted=2.22,
    extinction_coefficient_lewis=1.67,
    error_sample_length=0.001,
    error_sample_width=0.001,
    error_sample_mass=0.0001,
    surface_area=323.0, # OPTIONAL: surface area in m²/g
    error_surface_area=None,# OPTIONAL: error in surface area
)
data_handling.save_json(json_filename=f"{folder}.json")

***

## Gaussian Fitting

This notebook can fit two or three peaks in your respective Pyridine-adsorption IR spectra with a Gaussian. With the fitting parameters obtained from the fits, the number of active sites corresponding to Bronsted and/or Lewis sites can be calculated from

$N = \frac{S_{area} \cdot A_{peak}}{m_{sample} \cdot \varepsilon}$

With the sample area $S_{area}$, or area of the sample wafer, the sample mass $m_{sample}$, the absorption coeffiecient $\varepsilon$ of the investigated material and the peak area $A_{peak}$ which is obtained from the fitting parameters.

The calculated results can be saven to a TXT file.

Specify the folder `corr` which holds the corrected data which is to be fitted. 

In [ ]:
# Setup peak fitiing
corr = Path(folder) / "corr"

# Create fit configuration
fit_config = FitConfig(
    threshold=0.01
)

# Initialize PeakFitter
peak_fitter = PeakFitter(
    path_to_directory= cwd / corr,
    folder=corr,
    fit_config=fit_config
)
# Print available files
peak_fitter.available_files()


From the list above, choose a file by its index, which you would like to fit. The control plot shows, if all peaks that should be fitted are found. If peaks are missing or too many peaks are found by the algorithm, adjust the threshold value accordingly. Define a `name` which will be the file name of the saved plot. To control if all important peaks are found, check the control plot:

In [ ]:
# Add sample meta data
path_to_json = cwd / folder
peak_fitter.load_metadata_from_json(f"{path_to_json}.json")

# Extract data and find peaks
index = 4
name=f"{corr}_{index}"
peak_fitter.extract_data(index=index)
peak_fitter.get_control_plot(index=index)

In [ ]:
# does the Lewis peak have a shoulder?
peak_fitter.force_shoulder("lewis", force=False)

#Run the main analysis
peak_fitter.analyze()

# View results (area, shoulder, detection)
print(peak_fitter.print_results())
peak_fitter.plot_results(index=index)

---
Finally, with the data obtained from the Gaussians, calculate the number of active sites (in $\mu mol \cdot g^{-1}$) by providing the `sample_mass` in $g$ and the `abs_coeff` (absorption coefficent) in $cm \cdot \mu mol^{-1}$.

In [ ]:
# Calculate number of acid sites
n_sites = peak_fitter.calc_n_sites()
print("\nCalculated number of acid sites (mmol/g):")
print(f"Bronsted sites: {n_sites[0]:.3f} +/- {n_sites[1]} \u03BCmol/g")
print(f"Lewis sites: {n_sites[2]:.3f} +/- {n_sites[3]} \u03BCmol/g")

In [ ]:
# save results to existing json
peak_fitter.update_json(json_filename=f"{data_handling.folder}.json", index=index)